In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
import scipy.integrate
from scipy.special import erf
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
import re
# multi-core/thread:
import concurrent.futures

In [2]:
speed_pdf_dir = '../data/DM_Speed_PDF_extended_grid/'
speed_pdf_path = Path(speed_pdf_dir)
eta_func_dir = '../data/DM_eta_function_extended_grid/'
# 使用 iterdir() 遍历目录中的文件
speed_pdf_files = [file for file in speed_pdf_path.iterdir()]

In [3]:
def process_file(filename):
    file_stem = filename.stem
    speed_pdf = np.loadtxt(filename, skiprows = 1)
    speed_recipro = speed_pdf[:,1] / speed_pdf[:,0]
    eta_func = np.zeros((speed_pdf.shape[0],2))
    eta_func[:,0] = speed_pdf[:,0]
    for i in range(speed_pdf.shape[0]):
        eta_func[i,1] = scipy.integrate.simpson(speed_recipro[i:], x = speed_pdf[i:,0])
    np.savetxt(eta_func_dir + 'eta_func_' + file_stem[13:] + '.txt', eta_func)

In [4]:
def process_files_in_parallel(file_list):
    with concurrent.futures.ProcessPoolExecutor() as executor:
        executor.map(process_file, file_list)
process_files_in_parallel(speed_pdf_files)